In [26]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Flatten, Dense, Conv2D, MaxPool2D
import cv2
import numpy as np 
from tensorflow.keras.models import load_model

In [ ]:
#generators
train_ds = keras.utils.image_dataset_from_directory(
    directory ='C:/Users/bnk07/Downloads/mask/Face Mask Dataset/Train',
    labels = 'inferred',
    label_mode = 'int',
    batch_size= 32,
    image_size = (256, 256)
)
val_ds = keras.utils.image_dataset_from_directory(
    directory ='C:/Users/bnk07/Downloads/mask/Face Mask Dataset/Validation',
    labels = 'inferred',
    label_mode = 'int',    batch_size= 32,
    image_size = (256, 256)
)    

Found 10000 files belonging to 2 classes.
Found 800 files belonging to 2 classes.


In [28]:
#normalize
def process(image, label):
  image = tf.cast(image/255, tf.float32)
  return image, label
train_ds = train_ds.map(process)
val_ds = val_ds.map(process)

In [29]:
#CNN model
model = Sequential()
model.add(Conv2D(32, kernel_size = (3,3), padding = 'valid', activation = 'relu', input_shape=(256, 256, 3)))
model.add(MaxPool2D(pool_size=(2,2), strides = 2, padding = 'valid'))

model.add(Conv2D(64, kernel_size = (3,3), padding = 'valid', activation = 'relu'))
model.add(MaxPool2D(pool_size=(2,2), strides = 2, padding = 'valid'))

model.add(Conv2D(128, kernel_size = (3,3), padding = 'valid', activation = 'relu'))
model.add(MaxPool2D(pool_size=(2,2), strides = 2, padding = 'valid'))


model.add(Flatten())
model.add(Dense(128, activation = 'relu'))
model.add(Dense(64, activation = 'relu'))
model.add(Dense(1, activation = 'sigmoid'))


In [30]:
#summary
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,847,297 (56.64 MB)

 Trainable params: 14,847,297 (56.64 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
#compile
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [32]:
his = model.fit(train_ds, epochs  = 5, validation_data = val_ds)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 215s 680ms/step - accuracy: 0.9514 - loss: 0.1314 - val_accuracy: 0.9887 - val_loss: 0.0415
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 274s 719ms/step - accuracy: 0.9826 - loss: 0.0483 - val_accuracy: 0.9887 - val_loss: 0.0302
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 222s 710ms/step - accuracy: 0.9902 - loss: 0.0314 - val_accuracy: 0.9887 - val_loss: 0.0333
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 217s 692ms/step - accuracy: 0.9902 - loss: 0.0266 - val_accuracy: 0.9887 - val_loss: 0.0305
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 229s 730ms/step - accuracy: 0.9925 - loss: 0.0212 - val_accuracy: 0.9925 - val_loss: 0.0179


In [33]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    'C:/Users/bnk07/Downloads/mask/Face Mask Dataset/Test',
    image_size=(256, 256),   # Use the same image size as training
    batch_size=32,
    shuffle=False             # Important: don't shuffle for evaluation
)

Found 992 files belonging to 2 classes.


In [34]:
loss, accuracy = model.evaluate(test_ds)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

31/31 ━━━━━━━━━━━━━━━━━━━━ 18s 576ms/step - accuracy: 0.9667 - loss: 14.3902
Test Loss: 14.390209197998047
Test Accuracy: 0.9667338728904724


In [35]:
model.save("maskDetector.h5")
model = load_model("maskDetector.h5")

Testing from my own image input 

In [106]:
img = cv2.imread("C:/Users/bnk07/Downloads/mask/meumm.png")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (256, 256))
img = img / 255.0
img = np.expand_dims(img, axis=0) # shape: (1, 256, 256, 3)

In [107]:
pred = model.predict(img, verbose=0)
print(pred)
if pred[0][0] > 0.5:
    print("Unmasked")
else:
    
    print("Masked")

[[1.5755874e-11]]
Masked


Now ComputerVision input in RealTime

In [113]:
cap = cv2.VideoCapture(0)

In [114]:
# Load face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Convert to grayscale for face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Detect faces
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    
    for (x, y, w, h) in faces:
        # Crop the face
        face = frame[y:y+h, x:x+w]
        face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        face_resized = cv2.resize(face_rgb, (256, 256))
        face_resized = face_resized.astype("float32") / 255.0
        face_input = np.expand_dims(face_resized, axis=0)
        
        # Predict
        pred = model.predict(face_input, verbose=0)[0][0]
        
        # Decide label
        if pred > 0.5:
            label = "Unmasked"
            color = (0, 0, 255)  
        else:
            label = "Masked"
            color = (0, 255, 0)  
        
        # Draw rectangle around face
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        # Put label
        cv2.putText(frame, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
    
    # Show frame
    cv2.imshow("Mask Detection", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
